# Module 1 · Where the Memory and the Milliseconds Go

**NYU 2026** | Serving LLMs Efficiently at Scale | [Harshul Jain](https://github.com/harshuljain13)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/nyu-2026/demos/nyu_2026_module1_memory_and_latency.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/nyu-2026/demos/nyu_2026_module1_memory_and_latency.ipynb)

Seven questions, answered by measurement on whatever machine is running this cell:

| § | Question | Answer we measure |
|---|----------|-------------------|
| 01 | What did we just load? | Shape of the model, from `config.json` |
| 02 | Where did the memory go? | Weights are fixed; the rest is your serving budget |
| 03 | What fills the rest, and when does it stop fitting? | The KV cache — one K and one V vector per token, per user |
| 04 | How long before the user sees anything? | TTFT = one forward pass over the prompt |
| 05 | How long between tokens after that? | ITL, one pass per token, and the throughput gap |
| 06 | What does the cache do to each phase? | Prompt length moves TTFT; cache on/off moves ITL |
| 07 | What did all of it add up to? | Every number, in one table |

Pure PyTorch + Hugging Face. No vLLM, no external services, nothing simulated.

**The order matters.** §02 says how much memory is left, §03 says what claims it and how
fast, and only then do §04–§06 time the two phases that live inside that budget. Memory
first, milliseconds second.

**Hardware is detected, not assumed.** A CUDA GPU with ≥ 18 GB runs Mistral-7B; anything
smaller — including CPU-only Molab — falls back to a 1.1B model with smaller knobs. The
absolute numbers change with the machine; every shape in the notebook does not.

**Module 1 establishes the bottleneck; [Module 2](nyu_2026_module2_attention_and_quantization.ipynb)
attacks it.** §03 ends on a load that does not fit; Module 2 is how you make it fit.

## 00 · Setup

Installs, hardware detection, and the knobs every later section reads. Mistral-7B is a
gated repo, so if we are taking the 7B path this cell also checks your Hugging Face login
and says so now rather than failing part-way into a download.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers>=4.56', 'torch', 'accelerate',
                       'matplotlib', 'pandas', 'huggingface_hub'])

import gc, statistics, time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from matplotlib.colors import ListedColormap
from transformers import AutoModelForCausalLM, AutoTokenizer

GIB = 1024 ** 3

# Presenter-readable defaults: big text, no chartjunk.
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12, 'axes.titlesize': 14,
                     'axes.titleweight': 'bold', 'axes.spines.top': False,
                     'axes.spines.right': False})
COOL, WARM, GREY = '#2563eb', '#991b1b', '#cbd5e1'

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / GIB if HAS_CUDA else 0.0
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'

# One switch decides the model and every knob below it.
USE_7B = HAS_CUDA and VRAM_GIB >= 18
MODEL_ID = 'mistralai/Mistral-7B-v0.1' if USE_7B else 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32
ELEM_BYTES = torch.finfo(DTYPE).bits // 8

print(f'Device    : {GPU_NAME}' + (f'  ({VRAM_GIB:.1f} GiB)' if HAS_CUDA else ''))
print(f'Model     : {MODEL_ID}')
print(f'Precision : {str(DTYPE).replace("torch.", "")}  ({ELEM_BYTES} bytes per value)')
if not USE_7B:
    print('            ^ fallback path: same shapes, smaller numbers.')

In [ ]:
# Knobs. Scaled so that no cell takes more than ~30 s on its device class.
if USE_7B:
    # 5k prompt: long enough that prefill throughput is real work rather than
    # per-call overhead, which is what makes the tok/s in 04 worth quoting.
    PROMPT_TOKENS, DECODE_STEPS, UNCACHED_STEPS = 5000, 64, 16
    LENGTHS = [128, 512, 2048, 8192]
    CTX_MEASURED = [128, 512, 1024, 2048, 4096, 8192]
    BATCH_CTX, BATCH_MEASURED = 1024, [1, 2, 4, 8]
elif HAS_CUDA:
    PROMPT_TOKENS, DECODE_STEPS, UNCACHED_STEPS = 2048, 64, 16
    LENGTHS = [128, 512, 1024, 2048]
    CTX_MEASURED = [128, 256, 512, 1024, 2048]
    BATCH_CTX, BATCH_MEASURED = 512, [1, 2, 4, 8]
else:
    PROMPT_TOKENS, DECODE_STEPS, UNCACHED_STEPS = 512, 16, 8
    LENGTHS = [64, 128, 256, 512]
    CTX_MEASURED = [64, 128, 256, 512]
    BATCH_CTX, BATCH_MEASURED = 128, [1, 2, 4]

# §03 shows the real K/V tensors at this size: small enough to look at.
KV_DEMO_LEN = 48

# §03b-c are arithmetic, so these stay fixed on every device and stay comparable.
USERS = [1, 8, 32, 80]
CONTEXTS = [4096, 16384]

print(f'Prompt for 04  : {PROMPT_TOKENS} tokens')
print(f'Decode steps   : {DECODE_STEPS} cached, {UNCACHED_STEPS} uncached')
print(f'Length sweep   : {LENGTHS}')
print(f'Users measured : {BATCH_MEASURED} at {BATCH_CTX} tokens each')
print(f'Capacity grid  : {USERS} users x {[c // 1024 for c in CONTEXTS]}K context')

if USE_7B:
    try:
        from huggingface_hub import whoami
        print(f'HF token       : ok, logged in as {whoami()["name"]}')
    except Exception:
        print('HF token       : NOT FOUND. Run  huggingface-cli login  (or set HF_TOKEN),')
        print(f'                 then accept the licence at https://huggingface.co/{MODEL_ID}')
else:
    print('HF token       : not needed on the fallback path')

In [ ]:
def sync():
    if HAS_CUDA:
        torch.cuda.synchronize()


def timed(fn):
    """Wall time of fn() in seconds, device-synchronized on both sides."""
    sync()
    t0 = time.perf_counter()
    out = fn()
    sync()
    return out, time.perf_counter() - t0


def free_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


def kv_pairs(past):
    """(key, value) tensors per layer. transformers has shipped several cache
    classes, so normalize once here instead of in every cell below."""
    if hasattr(past, 'layers'):
        return [(lay.keys, lay.values) for lay in past.layers]
    if hasattr(past, 'key_cache'):
        return list(zip(past.key_cache, past.value_cache))
    return [(k, v) for k, v in past]


def kv_bytes(past):
    return sum(t.numel() * t.element_size() for pair in kv_pairs(past) for t in pair)


def kv_tokens(past):
    return int(kv_pairs(past)[0][0].shape[-2])


print('Helpers ready: timed, free_memory, kv_bytes, kv_tokens.')

## 01 · Model Loading

Six numbers from `config.json` decide everything that follows. The one to watch is the
**KV head count against the query head count** — where those differ, that is
grouped-query attention, and it is the only reason the arithmetic in §03 is survivable.

In [ ]:
print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model, load_sec = timed(lambda: AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map='auto' if HAS_CUDA else None).eval())
if not HAS_CUDA:
    model.to(DEVICE)

cfg = model.config
N_LAYERS = cfg.num_hidden_layers
N_HEADS = cfg.num_attention_heads
N_KV_HEADS = getattr(cfg, 'num_key_value_heads', None) or N_HEADS
HEAD_DIM = getattr(cfg, 'head_dim', None) or cfg.hidden_size // N_HEADS
GROUP = N_HEADS // N_KV_HEADS
N_PARAMS = sum(p.numel() for p in model.parameters())
WEIGHT_BYTES = sum(p.numel() * p.element_size() for p in model.parameters())

print(f'Load time      : {load_sec:.1f} s')
print(f'Layers         : {N_LAYERS}')
print(f'Hidden size    : {cfg.hidden_size}')
print(f'Query heads    : {N_HEADS}')
print(f'KV heads       : {N_KV_HEADS}' +
      (f'      <- GQA: {GROUP} query heads per KV head' if GROUP > 1 else '      (no GQA)'))
print(f'Head dim       : {HEAD_DIM}')
print(f'Parameters     : {N_PARAMS / 1e9:.2f} B')
print(f'Weights        : {WEIGHT_BYTES / GIB:.2f} GiB at {ELEM_BYTES} bytes each')

In [ ]:
def chart_attention_heads():
    """One square per head. Colour = which KV head it reads from."""
    colors = plt.cm.tab20(range(N_KV_HEADS))
    fig, ax = plt.subplots(figsize=(11, 3.2))

    for h in range(N_HEADS):
        ax.add_patch(plt.Rectangle((h, 1.2), 0.86, 0.8, color=colors[h // GROUP],
                                   ec='#000', lw=0.8))
    for g in range(N_KV_HEADS):
        ax.add_patch(plt.Rectangle((g * GROUP, 0), GROUP - 0.14, 0.8,
                                   color=colors[g], ec='#000', lw=1.4))

    ax.text(-0.6, 1.6, f'{N_HEADS} query heads', ha='right', va='center',
            fontweight='bold')
    ax.text(-0.6, 0.4, f'{N_KV_HEADS} KV heads', ha='right', va='center',
            fontweight='bold')
    ax.set_xlim(-N_HEADS * 0.3, N_HEADS)
    ax.set_ylim(-0.4, 2.4)
    ax.axis('off')
    ax.set_title(f'{GROUP} query heads per cached KV head\n'
                 f'{N_LAYERS} layers x {N_KV_HEADS} KV heads x {HEAD_DIM} dim '
                 f'is what we store', fontsize=13)
    plt.tight_layout()
    plt.show()


chart_attention_heads()

## 02 · Memory Distribution

Parameters times bytes-per-parameter. The weights are a **fixed** cost, paid once, shared
by every user. Everything left over is the budget you serve users out of — and on a GPU
we can read that remainder straight off the device.

In [ ]:
sync()
if HAS_CUDA:
    FREE_B, TOTAL_B = torch.cuda.mem_get_info()
    ALLOC_B = torch.cuda.memory_allocated()
    BUDGET_B, BUDGET_LABEL = FREE_B, 'measured free VRAM'
else:
    TOTAL_B = ALLOC_B = WEIGHT_BYTES
    FREE_B = 0
    # No VRAM to measure, so §03 needs a STATED budget. Labelled as an
    # assumption everywhere it is used, never as a measurement.
    BUDGET_B = 80 * GIB - WEIGHT_BYTES
    BUDGET_LABEL = 'ASSUMED: 80 GiB GPU minus these weights'

BUDGET_GIB = BUDGET_B / GIB

print(f'Weights               : {WEIGHT_BYTES / GIB:6.2f} GiB   ({WEIGHT_BYTES / 1e9:.1f} GB)')
if HAS_CUDA:
    print(f'Allocated on device   : {ALLOC_B / GIB:6.2f} GiB')
    print(f'Total device memory   : {TOTAL_B / GIB:6.2f} GiB')
print(f'Serving budget        : {BUDGET_GIB:6.2f} GiB   <- {BUDGET_LABEL}')
print()

WIDTH = 50
ref_b = TOTAL_B if HAS_CUDA else BUDGET_B + WEIGHT_BYTES
used_frac = (ref_b - BUDGET_B) / ref_b
filled = max(round(WIDTH * used_frac), 1)
print(f'  |{"W" * filled}{"." * (WIDTH - filled)}|  {ref_b / GIB:.1f} GiB')
print(f'   W = weights            {(ref_b - BUDGET_B) / GIB:6.2f} GiB  ({used_frac * 100:.0f}%)')
print(f'   . = free for KV cache  {BUDGET_GIB:6.2f} GiB  ({(1 - used_frac) * 100:.0f}%)')

In [ ]:
def chart_memory_split():
    used_gib = (ref_b - BUDGET_B) / GIB
    fig, ax = plt.subplots(figsize=(11, 2.6))

    ax.barh([0], [used_gib], color=COOL, ec='#000', label='weights + runtime')
    ax.barh([0], [BUDGET_GIB], left=[used_gib], color=GREY, ec='#000',
            label='free for KV cache')
    ax.text(used_gib / 2, 0, f'{used_gib:.1f} GiB', ha='center', va='center',
            color='white', fontweight='bold', fontsize=15)
    ax.text(used_gib + BUDGET_GIB / 2, 0, f'{BUDGET_GIB:.0f} GiB', ha='center',
            va='center', fontweight='bold', fontsize=15)

    ax.set_yticks([])
    ax.set_xlim(0, ref_b / GIB)
    ax.set_xlabel('memory (GiB)')
    ax.set_title(f'{GPU_NAME}: fixed cost vs serving budget, zero users connected')
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.75), ncol=2,
              frameon=False)
    plt.tight_layout()
    plt.show()


chart_memory_split()

## 03 · The KV Cache — What Fills the Rest

§02 left us a budget. This section is about the only thing that spends it.

Attention needs, for every token it has already seen, that token's **key** vector and
**value** vector. Recomputing them at every step would mean re-forwarding the whole
sequence every time, so they are computed once and kept. That store is the KV cache, and
its size is not a tuning parameter — it is a product:

$$\text{KV bytes} = \underbrace{2 \times L \times n_{kv} \times d_h \times \text{bytes}}_{\text{per token, fixed by architecture}}
\times \underbrace{\text{context length}}_{\text{03a}}
\times \underbrace{\text{concurrent users}}_{\text{03b}}$$

The leading 2 is one K plus one V. Note $n_{kv}$ is the **KV** head count, not the query
head count — that substitution is what GQA buys you. The weights were one number you pay
once; this is a product of three, and you control two of them.

First, look at the real tensors.

In [ ]:
ESSAY = ('The transformer architecture processes every token of the prompt in parallel '
         'during the forward pass, which makes prompt ingestion fast but generation '
         'strictly sequential. Each generated token must wait for the one before it. ')


def make_prompt(n_tokens, batch=1):
    """A prompt of exactly n_tokens, built by repeating ESSAY. batch > 1 stacks
    the same prompt as that many independent sequences."""
    text = ESSAY * (n_tokens // 20 + 2)
    ids = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=n_tokens)['input_ids']
    if batch > 1:
        ids = ids.repeat(batch, 1)
    return ids.to(DEVICE)


@torch.no_grad()
def prefill(ids, use_cache=True):
    """One forward pass over a whole prompt. Returns (model output, seconds)."""
    return timed(lambda: model(input_ids=ids, use_cache=use_cache))


print(f'Prompt builder ready against a {tokenizer.vocab_size:,}-token vocabulary.')
print('Used by every measurement from here down.')

In [ ]:
KV_DEMO_IDS = make_prompt(KV_DEMO_LEN)
KV_DEMO_OUT, _demo_sec = prefill(KV_DEMO_IDS)
KV_DEMO_CACHE = KV_DEMO_OUT.past_key_values
KV_DEMO_K, KV_DEMO_V = kv_pairs(KV_DEMO_CACHE)[0]        # layer 0
KV_DEMO_N = kv_tokens(KV_DEMO_CACHE)

KV_PER_TOKEN = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * ELEM_BYTES
KV_MEASURED_PER_TOKEN = kv_bytes(KV_DEMO_CACHE) / KV_DEMO_N

print(f'One prompt of {KV_DEMO_N} tokens, forwarded once with use_cache=True.')
print()
print('Layer 0 of the cache it left behind, exactly as it sits in memory:')
print(f'  keys   : {tuple(KV_DEMO_K.shape)}')
print(f'  values : {tuple(KV_DEMO_V.shape)}')
if KV_DEMO_K.shape[1] == N_KV_HEADS and KV_DEMO_K.shape[-2] == KV_DEMO_N:
    print(f'  axes   : (batch, kv_heads, tokens, head_dim)'
          f'  =  (1, {N_KV_HEADS}, {KV_DEMO_N}, {HEAD_DIM})')
else:
    print('  axes   : this model lays the cache out differently than (batch, heads,')
    print('           tokens, dim) - read the shapes above, not the labels below.')
print()
print(f'The tokens axis is the one that moves. One key vector and one value vector')
print(f'per token, each {HEAD_DIM} numbers long, in each of {N_KV_HEADS} KV heads, in each')
print(f'of {N_LAYERS} layers: add one token to the prompt and {2 * N_LAYERS * N_KV_HEADS} '
      f'new vectors get stored.')
print()
print('PREDICTED per token, from config.json alone')
print(f'  2 x {N_LAYERS} layers x {N_KV_HEADS} kv-heads x {HEAD_DIM} dim x {ELEM_BYTES} B'
      f'  =  {KV_PER_TOKEN:,} bytes  ({KV_PER_TOKEN / 1024:.0f} KiB)')
print('MEASURED, by weighing every tensor in the cache above')
print(f'  {kv_bytes(KV_DEMO_CACHE) / 1024 ** 2:,.1f} MiB over {KV_DEMO_N} tokens'
      f'  =  {KV_MEASURED_PER_TOKEN:,.0f} bytes per token')
print(f'  error vs prediction: '
      f'{abs(KV_MEASURED_PER_TOKEN - KV_PER_TOKEN) / KV_PER_TOKEN * 100:.2f}%')

In [ ]:
def chart_key_value_vectors():
    """The real layer-0 cache for one KV head. One row = one prompt token."""
    if KV_DEMO_K.ndim != 4:
        print(f'Cache layout is {tuple(KV_DEMO_K.shape)}, not (batch, heads, '
              f'tokens, dim) - nothing to slice for a heatmap.')
        return
    k = KV_DEMO_K[0, 0].detach().float().cpu().numpy()
    v = KV_DEMO_V[0, 0].detach().float().cpu().numpy()

    fig, (ax_k, ax_v) = plt.subplots(1, 2, figsize=(13, 4.8))
    for ax, mat, name in ((ax_k, k, 'keys'), (ax_v, v, 'values')):
        # Each panel gets its own scale: after RoPE the keys run much hotter
        # than the values, and a shared scale washes the values out to blank.
        lim = max(float(abs(mat).max()), 1e-6)
        im = ax.imshow(mat, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
        ax.set_xlabel(f'head dim ({mat.shape[1]} numbers per vector)')
        ax.set_ylabel('prompt token')
        ax.set_title(f'layer 0, KV head 0: {name}\n'
                     f'{mat.shape[0]} tokens x {mat.shape[1]} numbers, '
                     f'scaled to +/-{lim:.2f}')
        fig.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle(f'One row per token, kept for the whole request. '
                 f'x {N_KV_HEADS} KV heads x {N_LAYERS} layers x 2 = '
                 f'{KV_PER_TOKEN / 1024:.0f} KiB per token',
                 fontsize=13, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    plt.show()


chart_key_value_vectors()

### 03a · More prompt, more cache

Prefill at several lengths and **weigh the real cache each time** — no arithmetic. If the
per-token cost above is right, the measured points land on a straight line whose slope is
exactly that cost.

In [ ]:
def measure_cache_growth():
    """Prefill at each length and weigh the real cache. No arithmetic."""
    pairs = []
    for n_tok in CTX_MEASURED:
        ids = make_prompt(n_tok)
        out, _ = prefill(ids)
        pairs.append((int(ids.shape[1]), kv_bytes(out.past_key_values)))
        del out
        free_memory()
    return pairs


growth = measure_cache_growth()

growth_table = pd.DataFrame([
    {'Context tokens': n,
     'Cache measured (MiB)': round(b / 1024 ** 2, 1),
     'Cache predicted (MiB)': round(KV_PER_TOKEN * n / 1024 ** 2, 1),
     'Bytes per token': round(b / n)}
    for n, b in growth])

slope = (growth[-1][1] - growth[0][1]) / (growth[-1][0] - growth[0][0])
print(f'Measured slope : {slope:,.0f} bytes/token ({slope / 1024:.0f} KiB) '
      f'vs predicted {KV_PER_TOKEN / 1024:.0f} KiB')
print('Double the context, double the cache. Exactly linear - no compression,')
print('no decay, nothing that gets cheaper at scale.')

growth_table

In [ ]:
def chart_cache_linear():
    xs = [n for n, _ in growth]
    measured_gib = [b / GIB for _, b in growth]
    fig, ax = plt.subplots(figsize=(11, 4.4))

    ax.plot(xs, [KV_PER_TOKEN * n / GIB for n in xs], '-', color=GREY, lw=7,
            label=f'predicted: {KV_PER_TOKEN / 1024:.0f} KiB x context')
    ax.plot(xs, measured_gib, 'o-', color=COOL, ms=8, lw=2,
            label='measured cache')
    for x, gib in zip(xs, measured_gib):
        ax.annotate(f'{gib * 1024:.0f} MiB', (x, gib),
                    textcoords='offset points', xytext=(0, 11), ha='center',
                    fontsize=10)

    ax.set_xlabel('context length (tokens)')
    ax.set_ylabel('KV cache, ONE user (GiB)')
    ax.set_ylim(0, max(measured_gib) * 1.35)
    ax.set_title('One user: the cache is exactly linear in context length')
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()


chart_cache_linear()

### 03b · More users, more cache

The second direction. Every concurrent request carries its own keys and values, so users
multiply whatever 03a already measured. Prove it the same way: send the same prompt as
several independent sequences at once and weigh the cache again.

In [ ]:
def measure_batch_growth():
    """Prefill the same prompt as N independent sequences and weigh the cache.
    Stops at the first batch size this device refuses - that refusal is also a
    measurement."""
    rows, refused = [], None
    for n_seq in BATCH_MEASURED:
        try:
            ids = make_prompt(BATCH_CTX, batch=n_seq)
            out, _ = prefill(ids)
            rows.append((n_seq, int(ids.shape[1]), kv_bytes(out.past_key_values)))
            del out, ids
        except RuntimeError as exc:
            refused = (n_seq, str(exc).split('\n')[0][:88])
            break
        finally:
            free_memory()
    return rows, refused


batch_growth, batch_refused = measure_batch_growth()

batch_table = pd.DataFrame([
    {'Concurrent users': u,
     'Tokens each': n,
     'Cache measured (MiB)': round(b / 1024 ** 2, 1),
     'Cache predicted (MiB)': round(KV_PER_TOKEN * u * n / 1024 ** 2, 1),
     'Per user (MiB)': round(b / u / 1024 ** 2, 1)}
    for u, n, b in batch_growth])

if batch_growth:
    print(f'The same {batch_growth[0][1]}-token prompt, served to '
          f'1..{batch_growth[-1][0]} users at once:')
    print(f'  {batch_growth[0][2] / 1024 ** 2:,.1f} MiB for one user   ->   '
          f'{batch_growth[-1][2] / 1024 ** 2:,.1f} MiB for {batch_growth[-1][0]} users')
    print(f'  per-user cost stays at {batch_growth[-1][2] / batch_growth[-1][0] / 1024 ** 2:,.1f} '
          f'MiB, so the total is a plain multiplier')
else:
    print('No batch size fit on this device; the arithmetic below still holds.')
if batch_refused:
    print(f'  {batch_refused[0]} users did not fit here: {batch_refused[1]}')
print()
print('Note what we just paid for: the SAME text, prefilled once per user. Plain')
print('transformers shares nothing between sequences. A real engine can share the')
print('common prefix - that is Module 2 - but you cannot plan capacity on a')
print('discount your users may not give you.')

batch_table

In [ ]:
def chart_cache_vs_users():
    if not batch_growth:
        print('No measured batch points to plot.')
        return
    xs = [u for u, _, _ in batch_growth]
    measured = [b / 1024 ** 2 for _, _, b in batch_growth]
    predicted = [KV_PER_TOKEN * u * n / 1024 ** 2 for u, n, _ in batch_growth]

    fig, ax = plt.subplots(figsize=(11, 4.4))
    ax.plot(xs, predicted, '-', color=GREY, lw=7, label='predicted: per-user x users')
    ax.plot(xs, measured, 'o-', color=WARM, ms=8, lw=2, label='measured cache')
    for x, mib in zip(xs, measured):
        ax.annotate(f'{mib:,.0f} MiB', (x, mib), textcoords='offset points',
                    xytext=(0, 11), ha='center', fontsize=10)

    ax.set_xticks(xs)
    ax.set_xlabel('concurrent users (identical prompts)')
    ax.set_ylabel(f'KV cache at {batch_growth[0][1]} tokens each (MiB)')
    ax.set_ylim(0, max(measured) * 1.35)
    ax.set_title('Users multiply: nothing is shared between requests')
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()


chart_cache_vs_users()

### 03c · Both directions against the budget

Context and users multiply, so the honest question is not *how big is the cache* but
**where does each growth line cross the budget from §02**. Sweep context across, draw one
line per user count, and lay the budget across it.

In [ ]:
CTX_SWEEP = [512, 2048, 4096, 8192, 16384, 32768]
MODEL_MAX = getattr(cfg, 'max_position_embeddings', None)


def users_x_context():
    """KV GiB for every (users, context) pair, plus each row's budget crossing."""
    table = pd.DataFrame(
        [[f'{KV_PER_TOKEN * u * c / GIB:,.1f}' for c in CTX_SWEEP] for u in USERS],
        index=[f'{u} users' for u in USERS],
        columns=[f'{c // 1024}K' if c >= 1024 else str(c) for c in CTX_SWEEP])
    crossings = {u: BUDGET_B / (KV_PER_TOKEN * u) for u in USERS}
    return table, crossings


users_table, CROSSINGS = users_x_context()

print('KV cache in GiB, users (down) x context (across):')
print(f'Serving budget: {BUDGET_GIB:.1f} GiB  ({BUDGET_LABEL})')
print()
print('Longest context each user count can afford:')
for n_users, max_ctx in CROSSINGS.items():
    capped = ''
    if MODEL_MAX and max_ctx > MODEL_MAX:
        capped = f'   (model caps you at {MODEL_MAX:,} first)'
    label = f'{n_users} user' + ('' if n_users == 1 else 's')
    print(f'  {label:>9} -> {max_ctx:>9,.0f} tokens{capped}')

users_table

In [ ]:
def chart_growth_vs_budget():
    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13.5, 4.8))

    # Left: both directions at once, against the budget.
    for idx, n_users in enumerate(USERS):
        ys = [KV_PER_TOKEN * n_users * c / GIB for c in CTX_SWEEP]
        ax_l.plot(CTX_SWEEP, ys, 'o-', ms=5, lw=2,
                  color=plt.cm.viridis(idx / max(len(USERS) - 1, 1)),
                  label=f'{n_users} users')
        cross = CROSSINGS[n_users]
        if CTX_SWEEP[0] <= cross <= CTX_SWEEP[-1]:
            ax_l.plot([cross], [BUDGET_GIB], 'X', ms=14, color=WARM, zorder=5)
            ax_l.annotate(f'{cross / 1024:.1f}K', (cross, BUDGET_GIB),
                          textcoords='offset points', xytext=(6, 10),
                          fontsize=11, fontweight='bold', color=WARM)

    ax_l.axhline(BUDGET_GIB, color='#166534', ls='--', lw=2.5,
                 label=f'budget = {BUDGET_GIB:.0f} GiB')
    if MODEL_MAX and CTX_SWEEP[0] <= MODEL_MAX <= CTX_SWEEP[-1]:
        ax_l.axvline(MODEL_MAX, color='#475569', ls=':', lw=1.8)
        ax_l.text(MODEL_MAX, BUDGET_GIB * 2.05, ' model limit', fontsize=10,
                  color='#475569')

    ax_l.set_xscale('log', base=2)
    ax_l.set_xticks(CTX_SWEEP)
    ax_l.set_xticklabels([f'{c // 1024}K' if c >= 1024 else str(c)
                          for c in CTX_SWEEP])
    ax_l.set_ylim(0, BUDGET_GIB * 2.2)
    ax_l.set_xlabel('context length')
    ax_l.set_ylabel('KV cache (GiB)')
    ax_l.set_title('X marks where you run out of memory')
    ax_l.legend(fontsize=9, loc='upper left')

    # Right: the same thing as a fits/OOM grid at the two contexts that matter.
    grid = [[KV_PER_TOKEN * u * c / GIB for c in CONTEXTS] for u in USERS]
    fits = [[0 if v <= BUDGET_GIB else 1 for v in row] for row in grid]
    ax_r.imshow(fits, cmap=ListedColormap(['#dcfce7', '#fecaca']), vmin=0,
                vmax=1, aspect='auto')
    for r_i, row in enumerate(grid):
        for c_i, val in enumerate(row):
            tag = '' if val <= BUDGET_GIB else '\nOOM'
            ax_r.text(c_i, r_i, f'{val:,.1f} GiB{tag}', ha='center',
                      va='center', fontsize=12, fontweight='bold',
                      color='#166534' if val <= BUDGET_GIB else '#991b1b')
    ax_r.set_xticks(range(len(CONTEXTS)),
                    [f'{c // 1024}K context' for c in CONTEXTS])
    ax_r.set_yticks(range(len(USERS)), [f'{u} users' for u in USERS])
    ax_r.set_title(f'Against {BUDGET_GIB:.0f} GiB of budget')

    plt.tight_layout()
    plt.show()


chart_growth_vs_budget()

In [ ]:
def capacity_at_contexts():
    rows = []
    for c in CONTEXTS:
        per_user = KV_PER_TOKEN * c / GIB
        rows.append({'Context': f'{c // 1024}K tokens',
                     'KV per user (GiB)': round(per_user, 3),
                     'Users that fit': int(BUDGET_GIB / per_user),
                     'At 80 users (GiB)': round(per_user * 80, 1),
                     '80 users?': 'fits' if per_user * 80 <= BUDGET_GIB else 'OOM'})
    return pd.DataFrame(rows)


MAX_USERS = capacity_at_contexts()

worst = max(CONTEXTS)
need_gib = KV_PER_TOKEN * worst * 80 / GIB
print(f'Budget: {BUDGET_GIB:.1f} GiB  ({BUDGET_LABEL})')
print()
if need_gib > BUDGET_GIB:
    print(f'80 users at {worst // 1024}K context needs {need_gib:,.0f} GiB '
          f'= {need_gib / BUDGET_GIB:.1f}x what we have.')
    print('Nothing about that is fixable by waiting for a faster GPU. The only')
    print('moves are to shrink a term in the product - which is Module 2.')
else:
    print(f'80 users at {worst // 1024}K context needs {need_gib:,.0f} GiB, which '
          f'fits in {BUDGET_GIB:.0f} GiB -')
    print('only because this fallback model carries a far smaller cache than a 7B.')
    print('The next cell runs the same arithmetic at production scale, where it')
    print('does not fit. Shrinking a term in the product is Module 2.')

MAX_USERS

In [ ]:
# On the fallback path the numbers above are real but small. The
# production-shaped version of the same arithmetic is worth seeing either way,
# so compute it from Mistral-7B's PUBLISHED config and label it as derived.
REF_NAME = 'Mistral-7B-v0.1 @ bf16 on an 80 GiB GPU'
REF_PER_TOKEN = 2 * 32 * 8 * 128 * 2          # layers, kv-heads, head_dim, bytes
REF_BUDGET_GIB = 80.0 - 13.5                  # minus its weights

if USE_7B:
    print('Measured above IS the reference configuration - nothing to add.')
else:
    here = MODEL_ID.split('/')[-1]
    print(f'DERIVED, not measured here: {REF_NAME}')
    print(f'  KV per token   : {REF_PER_TOKEN / 1024:.0f} KiB'
          f'   (vs {KV_PER_TOKEN / 1024:.0f} KiB measured on {here})')
    print(f'  Serving budget : {REF_BUDGET_GIB:.1f} GiB')
    for ref_users, ctx in [(80, 4096), (80, 16384)]:
        gib = REF_PER_TOKEN * ref_users * ctx / GIB
        verdict = ('fits' if gib <= REF_BUDGET_GIB
                   else f'OOM, {gib / REF_BUDGET_GIB:.1f}x the GPU')
        print(f'  {ref_users} users x {ctx // 1024}K = {gib:5.0f} GiB   -> {verdict}')
    print()
    print('Same shape, bigger numbers. This is the arithmetic the slides quote,')
    print('and it comes from config.json - which is the whole point of 03.')

## 04 · Prefill

Memory is settled. Now the clock.

**Prefill** is one forward pass over the whole prompt. Every token is processed in
parallel, the KV cache for all of them is written in that single pass, and the first
output token falls out of the end of it — so prefill *is* time-to-first-token. Warm up
once (the first call compiles kernels), then time one pass.

In [ ]:
PROMPT = make_prompt(PROMPT_TOKENS)
N_PROMPT = PROMPT.shape[1]

prefill(PROMPT)                      # warm-up, not timed
prefill_out, prefill_sec = prefill(PROMPT)
TTFT_MS = prefill_sec * 1000
PREFILL_TPS = N_PROMPT / prefill_sec

print(f'Prompt            : {N_PROMPT} tokens')
print(f'TTFT              : {TTFT_MS:.0f} ms      <- one forward pass')
print(f'Per input token   : {TTFT_MS / N_PROMPT:.3f} ms')
print(f'Prefill throughput: {PREFILL_TPS:,.0f} tokens/s')
print(f'Cache written     : {kv_tokens(prefill_out.past_key_values):,} tokens, '
      f'{kv_bytes(prefill_out.past_key_values) / 1024 ** 2:,.0f} MiB  '
      f'(§03 priced this at {KV_PER_TOKEN / 1024:.0f} KiB/token)')

In [ ]:
def chart_ttft_timeline():
    fig, ax = plt.subplots(figsize=(11, 2.8))

    ax.barh([0], [TTFT_MS], color=COOL, ec='#000', height=0.5)
    ax.text(TTFT_MS / 2, 0, f'prefill: {N_PROMPT} tokens in ONE pass',
            ha='center', va='center', color='white', fontweight='bold',
            fontsize=13)
    ax.annotate(f'first token\n{TTFT_MS:.0f} ms', xy=(TTFT_MS, 0),
                xytext=(TTFT_MS * 1.08, 0), va='center', fontsize=13,
                fontweight='bold', color=WARM,
                arrowprops=dict(arrowstyle='->', color=WARM, lw=1.6))
    ax.text(0, -0.45, 'user sees nothing during this window', fontsize=11,
            color='#475569')

    ax.set_yticks([])
    ax.set_xlim(0, TTFT_MS * 1.45)
    ax.set_xlabel('time (ms)')
    ax.set_title(f'Time to first token at {PREFILL_TPS:,.0f} input tokens/s')
    plt.tight_layout()
    plt.show()


chart_ttft_timeline()

## 05 · Decode

After prefill, every further token is its own forward pass with **one** input token — the
earlier keys and values come from `past_key_values` instead of being recomputed. The gap
between tokens is inter-token latency (ITL).

One pass moved the whole prompt. Now one pass moves one token. Watch what that does to
throughput.

In [ ]:
@torch.no_grad()
def decode_cached(prompt_ids, steps):
    """Generate `steps` tokens one at a time, reusing the KV cache.
    Returns per-step latency in ms."""
    out, _ = prefill(prompt_ids)
    past = out.past_key_values
    tok = out.logits[:, -1:].argmax(-1)

    per_step = []
    for _ in range(steps):
        out, sec = timed(lambda: model(input_ids=tok, past_key_values=past,
                                       use_cache=True))
        past, tok = out.past_key_values, out.logits[:, -1:].argmax(-1)
        per_step.append(sec * 1000)
    return per_step, past


itl_cached, FINAL_CACHE = decode_cached(PROMPT, DECODE_STEPS)

ITL_MEAN = statistics.mean(itl_cached)
ITL_P50 = statistics.median(itl_cached)
DECODE_TPS = 1000 / ITL_MEAN

print(f'Decoded           : {DECODE_STEPS} tokens, one forward pass each')
print(f'ITL mean          : {ITL_MEAN:.1f} ms')
print(f'ITL p50           : {ITL_P50:.1f} ms')
print(f'Decode throughput : {DECODE_TPS:.1f} tokens/s  (single stream)')
print()
print(f'Prefill moved {N_PROMPT} tokens at {PREFILL_TPS:,.0f} tok/s.')
print(f'Decode moves {DECODE_STEPS} tokens at {DECODE_TPS:.1f} tok/s.')
print(f'Same model, same GPU, {PREFILL_TPS / DECODE_TPS:,.0f}x apart.')
print()
print(f'Cache now holds    : {kv_tokens(FINAL_CACHE):,} tokens, '
      f'{kv_bytes(FINAL_CACHE) / 1024 ** 2:,.0f} MiB '
      f'(it grew by one token per step)')

In [ ]:
def chart_itl_steps():
    fig, ax = plt.subplots(figsize=(11, 4.2))

    ax.plot(range(1, DECODE_STEPS + 1), itl_cached, 'o-', color=COOL, ms=4,
            lw=1.6, label='measured per step')
    ax.axhline(ITL_P50, color=WARM, ls='--', lw=2, label=f'p50 = {ITL_P50:.1f} ms')
    ax.axhline(ITL_MEAN, color='#166534', ls=':', lw=2,
               label=f'mean = {ITL_MEAN:.1f} ms')
    ax.set_ylim(0, max(itl_cached) * 1.3)
    ax.set_xlabel('decode step')
    ax.set_ylabel('inter-token latency (ms)')
    ax.set_title(f'{DECODE_STEPS} tokens with the KV cache: flat at '
                 f'{ITL_P50:.0f} ms/token, {DECODE_TPS:.0f} tok/s')
    ax.legend()
    plt.tight_layout()
    plt.show()


chart_itl_steps()

In [ ]:
def chart_prefill_vs_decode():
    """The same hardware, measured twice: throughput, then cost per token."""
    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.4))

    ax_l.bar(['prefill\n(input tokens)', 'decode\n(output tokens)'],
             [PREFILL_TPS, DECODE_TPS], color=[COOL, WARM], ec='#000', width=0.5)
    ax_l.set_yscale('log')
    ax_l.set_ylim(1, PREFILL_TPS * 6)
    for pos, val in enumerate([PREFILL_TPS, DECODE_TPS]):
        ax_l.text(pos, val * 1.15, f'{val:,.0f} tok/s', ha='center',
                  va='bottom', fontsize=14, fontweight='bold')
    ax_l.set_ylabel('tokens/s (log scale)')
    ax_l.set_title(f'{PREFILL_TPS / DECODE_TPS:,.0f}x throughput gap')

    per_in = TTFT_MS / N_PROMPT
    ax_r.bar(['per input token', 'per output token'], [per_in, ITL_MEAN],
             color=[COOL, WARM], ec='#000', width=0.5)
    for pos, val in enumerate([per_in, ITL_MEAN]):
        ax_r.text(pos, val, f'{val:.2f} ms', ha='center', va='bottom',
                  fontsize=14, fontweight='bold')
    ax_r.set_ylim(0, ITL_MEAN * 1.3)
    ax_r.set_ylabel('milliseconds')
    ax_r.set_title('Output tokens are the expensive ones')

    fig.suptitle('Prefill is compute-bound and parallel; decode is memory-bound '
                 'and sequential', fontsize=13, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()


chart_prefill_vs_decode()

## 06 · What the KV Cache Does to Each Phase

Two phases, one cache, and two levers worth knowing — because each lever moves exactly
one of the phases:

| Lever | Prefill (TTFT) | Decode (ITL) |
|---|---|---|
| **06a** make the prompt longer | rises, roughly in proportion | barely moves |
| **06b** turn the cache off | barely moves | collapses |

Both are measured below. Neither is obvious, and quoting one average latency hides both.

### 06a · A longer prompt

Four prompt lengths. For each: TTFT from one prefill, and mean ITL over 32 decoded
tokens. §03a already weighed the cache across this same range and found it linear — so
this is the same lever, seen on the clock instead of the scale. Prefill has to compute
and store every one of those tokens before it can answer; decode then reads the result
one token at a time, which costs the same whether the cache behind it is short or long.

In [ ]:
def measure_prompt_lengths():
    """TTFT from one prefill, plus mean ITL over 32 tokens, per length."""
    rows = []
    for n_tok in LENGTHS:
        ids = make_prompt(n_tok)
        prefill(ids)                              # warm-up at this shape
        _, sec = prefill(ids)
        steps, _cache = decode_cached(ids, 32)
        rows.append({'Prompt tokens': int(ids.shape[1]),
                     'KV cache predicted (MiB)': round(KV_PER_TOKEN * int(ids.shape[1]) / 1024 ** 2, 1),
                     'TTFT (ms)': round(sec * 1000, 1),
                     'ms per input token': round(sec * 1000 / ids.shape[1], 3),
                     'ITL mean (ms)': round(statistics.mean(steps), 1),
                     'Decode tok/s': round(1000 / statistics.mean(steps), 1)})
        free_memory()
    return pd.DataFrame(rows)


length_table = measure_prompt_lengths()

ttft_growth = length_table['TTFT (ms)'].iloc[-1] / length_table['TTFT (ms)'].iloc[0]
itl_growth = length_table['ITL mean (ms)'].iloc[-1] / length_table['ITL mean (ms)'].iloc[0]
print(f'{LENGTHS[0]} -> {LENGTHS[-1]} tokens is {LENGTHS[-1] // LENGTHS[0]}x more prompt,')
print(f'and {LENGTHS[-1] // LENGTHS[0]}x more KV cache to compute and store.')
print(f'  TTFT grew {ttft_growth:7.1f}x')
print(f'  ITL  grew {itl_growth:7.1f}x')
print(f'  -> the same lever moves TTFT {ttft_growth / itl_growth:.0f}x harder than it')
print('     moves ITL. That ratio is the point, not either number alone.')

length_table

In [ ]:
def chart_length_sweep():
    xs = length_table['Prompt tokens']
    ttft = length_table['TTFT (ms)']
    itl = length_table['ITL mean (ms)']
    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, 4.4))

    ax_l.plot(xs, ttft, 'o-', color=COOL, ms=7, lw=2)
    for x, y in zip(xs, ttft):
        ax_l.annotate(f'{y:.0f} ms', (x, y), textcoords='offset points',
                      xytext=(0, 9), ha='center', fontsize=11)
    ax_l.set_ylim(0, ttft.max() * 1.3)
    ax_l.set_xlabel('prompt tokens')
    ax_l.set_ylabel('TTFT (ms)')
    ax_l.set_title(f'TTFT scales with the prompt ({ttft_growth:.0f}x)')

    ax_r.plot(xs, itl, 'o-', color=WARM, ms=7, lw=2)
    for x, y in zip(xs, itl):
        ax_r.annotate(f'{y:.1f} ms', (x, y), textcoords='offset points',
                      xytext=(0, 9), ha='center', fontsize=11)
    ax_r.set_ylim(0, itl.max() * 1.8)
    ax_r.set_xlabel('prompt tokens')
    ax_r.set_ylabel('ITL (ms)')
    ax_r.set_title('ITL barely notices' if itl_growth < 1.5
                   else f'ITL moves {itl_growth:.1f}x over the same range')

    for axis in (ax_l, ax_r):
        axis.set_xscale('log', base=2)
        axis.set_xticks(LENGTHS)
        axis.set_xticklabels([str(v) for v in LENGTHS])

    plt.tight_layout()
    plt.show()


chart_length_sweep()

### 06b · The cache, switched off

`use_cache=False` on both phases, so the comparison is symmetric.

Prefill computes the same attention either way — the only difference is whether the keys
and values it produced are *kept*. Decode without them has to re-forward the entire
sequence to generate each next token. Same switch, two very different bills.

Expect prefill to come out within a few percent either way, and possibly a little
*faster* without the cache: it still computes every K and V, it just skips storing
them. On a memory-bound machine those stores are visible; the compute decode would
have to redo is not in the same league.

In [ ]:
@torch.no_grad()
def decode_uncached(prompt_ids, steps):
    """Same output, no cache: re-forward the full sequence every step."""
    out, _ = prefill(prompt_ids, use_cache=False)
    seq = torch.cat([prompt_ids, out.logits[:, -1:].argmax(-1)], dim=1)

    per_step = []
    for _ in range(steps):
        out, sec = timed(lambda: model(input_ids=seq, use_cache=False))
        seq = torch.cat([seq, out.logits[:, -1:].argmax(-1)], dim=1)
        per_step.append(sec * 1000)
    return per_step


def median_ttft(use_cache, repeats=3):
    """Prefill the same prompt a few times and take the median. The two settings
    are measured back to back, here, so they are comparable to each other -
    §04's number was taken on a differently-loaded device."""
    prefill(PROMPT, use_cache=use_cache)              # warm-up at this shape
    return statistics.median(
        prefill(PROMPT, use_cache=use_cache)[1] * 1000 for _ in range(repeats))


free_memory()
itl_uncached = decode_uncached(PROMPT, UNCACHED_STEPS)
ITL_NOCACHE_MEAN = statistics.mean(itl_uncached)

free_memory()
TTFT_CACHE_MS = median_ttft(True)
TTFT_NOCACHE_MS = median_ttft(False)
TTFT_DELTA_PCT = (TTFT_NOCACHE_MS - TTFT_CACHE_MS) / TTFT_CACHE_MS * 100

print('DECODE - the cache is the whole design')
print(f'  with cache    : {ITL_MEAN:8.1f} ms/token mean, flat')
print(f'  without cache : {ITL_NOCACHE_MEAN:8.1f} ms/token mean, '
      f'{itl_uncached[0]:.0f} -> {itl_uncached[-1]:.0f} ms across {UNCACHED_STEPS} steps')
print(f'  the cache is worth {ITL_NOCACHE_MEAN / ITL_MEAN:.0f}x at {N_PROMPT} tokens of')
print('  context, and more at every longer one')
print()
print('PREFILL - the same attention either way (median of 3 passes each)')
print(f'  with cache    : {TTFT_CACHE_MS:8.1f} ms')
print(f'  without cache : {TTFT_NOCACHE_MS:8.1f} ms')
print(f'  difference    : {TTFT_DELTA_PCT:+8.1f}%')
if TTFT_DELTA_PCT < -5:
    print('  ^ a little cheaper without it. Prefill still computes every K and V;')
    print('    it just skips STORING them. Saving the stores is worth a few percent.')
    print('    Decode would have to redo the compute - which is worth 10-100x.')
elif abs(TTFT_DELTA_PCT) > 15:
    print('  ^ that gap is larger than the effect being measured. This device is')
    print('    noisy at this prompt size; re-run, or read the ratio in DECODE only.')
else:
    print('  ^ within noise: prefill pays essentially nothing to fill the cache.')
print()
print(f'One switch, two bills: decode {ITL_NOCACHE_MEAN / ITL_MEAN:.0f}x worse, '
      f'prefill {TTFT_DELTA_PCT:+.1f}%.')
print('That asymmetry is why §03 came first: the thing you spend the whole memory')
print('budget on is the thing that makes generation possible at all.')

In [ ]:
def chart_cache_on_off():
    n_un = len(itl_uncached)
    mean_on = statistics.mean(itl_cached[:n_un])
    mean_off = ITL_NOCACHE_MEAN

    fig, (ax_l, ax_m, ax_r) = plt.subplots(1, 3, figsize=(15, 4.4))

    ax_l.plot(range(1, n_un + 1), itl_cached[:n_un], 'o-', color=COOL, ms=5,
              lw=1.8, label='use_cache=True')
    ax_l.plot(range(1, n_un + 1), itl_uncached, 'o-', color=WARM, ms=5, lw=1.8,
              label='use_cache=False')
    ax_l.set_ylim(0, mean_off * 1.6)
    ax_l.set_xlabel('decode step')
    ax_l.set_ylabel('ITL (ms)')
    ax_l.set_title(f'Same {n_un} tokens, on vs off', fontsize=12)
    ax_l.legend(fontsize=10)

    ax_m.bar(['with cache', 'no cache'], [mean_on, mean_off],
             color=[COOL, WARM], ec='#000', width=0.55)
    for pos, val in enumerate([mean_on, mean_off]):
        ax_m.text(pos, val, f'{val:.0f} ms', ha='center', va='bottom',
                  fontsize=14, fontweight='bold')
    ax_m.set_ylim(0, mean_off * 1.3)
    ax_m.set_ylabel('mean ms per OUTPUT token')
    ax_m.set_title(f'DECODE: {mean_off / mean_on:.0f}x slower without it',
                   fontsize=12)

    ax_r.bar(['with cache', 'no cache'], [TTFT_CACHE_MS, TTFT_NOCACHE_MS],
             color=[COOL, GREY], ec='#000', width=0.55)
    for pos, val in enumerate([TTFT_CACHE_MS, TTFT_NOCACHE_MS]):
        ax_r.text(pos, val, f'{val:.0f} ms', ha='center', va='bottom',
                  fontsize=14, fontweight='bold')
    ax_r.set_ylim(0, max(TTFT_CACHE_MS, TTFT_NOCACHE_MS) * 1.35)
    ax_r.set_ylabel('time to first token (ms)')
    ax_r.set_title(f'PREFILL: {TTFT_DELTA_PCT:+.1f}%', fontsize=12)

    fig.suptitle(f'One switch, two bills: decode {mean_off / mean_on:.0f}x worse, '
                 f'prefill {TTFT_DELTA_PCT:+.1f}%',
                 fontsize=13, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()


chart_cache_on_off()

## 07 · Summary — Every Number We Measured

One machine, no simulation. The only predicted value in the notebook (KV bytes per token)
was weighed against the real cache three times: against the tensors in §03, as a slope
over context in §03a, and as a multiplier over users in §03b.

In [ ]:
# 06b is optional, so fall back cleanly if the room skipped it.
no_cache_itl = (f'{ITL_NOCACHE_MEAN:.1f} ms'
                if 'ITL_NOCACHE_MEAN' in globals() else 'not run')
no_cache_ttft = (f'{TTFT_NOCACHE_MS:.0f} ms'
                 if 'TTFT_NOCACHE_MS' in globals() else 'not run')
cache_ttft = (f'{TTFT_CACHE_MS:.0f} ms'
              if 'TTFT_CACHE_MS' in globals() else 'not run')
batch_measured = (f'{batch_growth[-1][2] / 1024 ** 2:,.0f} MiB'
                  if batch_growth else 'not run')
batch_measured_note = (f'{batch_growth[-1][0]} users x {batch_growth[-1][1]} tokens, '
                       f'weighed' if batch_growth else 'no batch size fit')

summary = pd.DataFrame([
    ('01  Model',    'Parameters',                   f'{N_PARAMS / 1e9:.2f} B',
     f'{MODEL_ID.split("/")[-1]} at {ELEM_BYTES} bytes/value'),
    ('01  Model',    'Layers x KV heads x head dim', f'{N_LAYERS} x {N_KV_HEADS} x {HEAD_DIM}',
     f'GQA: {GROUP} query heads per KV head'),
    ('02  Memory',   'Weights',                      f'{WEIGHT_BYTES / GIB:.2f} GiB',
     'fixed cost, shared by all users'),
    ('02  Memory',   'Serving budget',               f'{BUDGET_GIB:.1f} GiB',
     BUDGET_LABEL),
    ('03  KV cache', 'KV cache per token',           f'{KV_PER_TOKEN / 1024:.0f} KiB',
     f'predicted and measured, {abs(KV_MEASURED_PER_TOKEN - KV_PER_TOKEN) / KV_PER_TOKEN * 100:.2f}% apart'),
    ('03a Context',  'Measured slope vs context',    f'{slope / 1024:.0f} KiB/token',
     f'linear across {CTX_MEASURED[0]} -> {CTX_MEASURED[-1]} tokens'),
    ('03b Users',    'Measured cache vs users',      batch_measured,
     batch_measured_note),
    ('03b Users',    'Longest context at 80 users',  f'{CROSSINGS[80] / 1024:,.1f}K tokens',
     'where the 80-user line crosses the budget'),
    ('03c Capacity', f'Users at {CONTEXTS[0] // 1024}K context',
     f'{int(BUDGET_GIB / (KV_PER_TOKEN * CONTEXTS[0] / GIB))}',
     f'{KV_PER_TOKEN * CONTEXTS[0] / GIB:.2f} GiB each'),
    ('03c Capacity', f'Users at {CONTEXTS[-1] // 1024}K context',
     f'{int(BUDGET_GIB / (KV_PER_TOKEN * CONTEXTS[-1] / GIB))}',
     f'{KV_PER_TOKEN * CONTEXTS[-1] / GIB:.2f} GiB each'),
    ('04  Prefill',  f'TTFT, {N_PROMPT}-token prompt', f'{TTFT_MS:.0f} ms',
     f'{PREFILL_TPS:,.0f} input tok/s, one pass'),
    ('05  Decode',   'ITL mean / p50',               f'{ITL_MEAN:.1f} / {ITL_P50:.1f} ms',
     f'{DECODE_TPS:.0f} tok/s, single stream'),
    ('05  Decode',   'Prefill vs decode throughput', f'{PREFILL_TPS / DECODE_TPS:,.0f}x',
     'parallel pass vs one pass per token'),
    ('06a Length',   f'TTFT, {LENGTHS[0]} -> {LENGTHS[-1]} tokens', f'{ttft_growth:.0f}x',
     'prefill is length-proportional'),
    ('06a Length',   f'ITL, {LENGTHS[0]} -> {LENGTHS[-1]} tokens',  f'{itl_growth:.2f}x',
     f'vs {ttft_growth:.0f}x for TTFT over the same range'),
    ('06b Cache off', 'ITL without KV cache',        no_cache_itl,
     'and climbing with every step'),
    ('06b Cache off', 'TTFT with KV cache, re-timed', cache_ttft,
     'median of 3, measured against the row below'),
    ('06b Cache off', 'TTFT without KV cache',       no_cache_ttft,
     f'{TTFT_DELTA_PCT:+.1f}% vs with it - prefill does the same work'),
], columns=['Section', 'Measurement', 'Value', 'What it means'])


summary

### What we discussed, in one table

| What we found | Why it is true | What it forces you to do |
|---|---|---|
| The weights are a fixed, one-time cost | Parameters x bytes-per-parameter, loaded once | Pick the GPU from the weights; plan capacity from what is left |
| The free remainder is the whole serving budget | Nothing else shares it | Every user you add is a claim on one number |
| The cache stores one K and one V vector per token | Read straight off `past_key_values`, layer by layer | `2 x L x n_kv x d_h x bytes` is a price you can quote before you deploy |
| KV bytes/token matches the config formula exactly | Predicted from `config.json`, weighed against the real tensors | Capacity planning is arithmetic, not benchmarking |
| **The cache is linear in context** | Measured across a 64x range — no compression, no decay | Context length is a capacity decision, not a product one |
| **The cache is linear in users too** | Measured by batching: per-user cost never drops | The two multiply — doubling both quadruples the cache |
| The 80-user line crosses the budget early | Budget / (bytes-per-token x context) | Concurrency and context trade directly against each other |
| TTFT is one parallel pass | Prefill processes all prompt tokens at once, writing the whole cache | Long prompts cost latency *before* the first character |
| Decode runs orders of magnitude slower than prefill | One forward pass per output token, sequential by construction | Output tokens are the expensive ones; price and SLO them separately |
| A longer prompt moves TTFT far harder than it moves ITL | Prefill is compute-shaped, decode is memory-shaped | Two phases, two bottlenecks — never quote one average |
| Switching the cache off wrecks ITL and leaves TTFT alone | Prefill does the same work either way; decode would have to re-forward everything | The KV cache is not an optimization, it is the design |

---

**Every number in this module, measured. Next: how to make the load fit.**

§03 is the handoff: the load we want needs more KV cache than we have. Two terms in that
product are attackable — the KV cache per token (GQA → MQA → MLA) and the weights
(INT8 → INT4). That is [Module 2](nyu_2026_module2_attention_and_quantization.ipynb).